In [8]:
state = dict()

state.update({"name":"fdq", "age": "20"})
print(state)

state.setdefault("errors", []).append("error")

ls = list()
ls.append("hello")
ls.append(1)
print(ls)

print(state)


{'name': 'fdq', 'age': '20'}
['hello', 1]
{'name': 'fdq', 'age': '20', 'errors': ['error']}


In [2]:
import asyncio
import nest_asyncio
import uvicorn

from fastapi import FastAPI
from fastapi.responses import StreamingResponse

nest_asyncio.apply()

app = FastAPI()


async def event_stream():
    for i in range(1, 6):
        msg = (
            f"id: {i}\n"
            f"event: message\n"
            f"data: hello {i}\n\n"
        )
        yield msg
        await asyncio.sleep(1)


@app.get("/sse")
async def sse():
    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
        }
    )


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=9000,
    log_level="debug"
)

server = uvicorn.Server(config)

task = asyncio.create_task(server.serve())